# End-to-End Piano Roll Generation

Full pipeline: **Flow model** (L0-L3) → **Inverse PCA** → **HMEP** (L4-L5 prediction) → **Decoder** → Piano roll

Run this notebook on **lecun** where all checkpoints and PCA models live.

In [ ]:
import os, pickle, torch, numpy as np, matplotlib.pyplot as plt
from pathlib import Path
import torch.nn.functional as F

from midi_rae.core import PatchState, HierarchicalPatchState, EncoderOutput
from midi_rae.swin import SwinDecoder, SwinMaskedEmbeddingPredictor, SwinEncoder
from midi_rae.train_flow import CrossLevelFlowModel, PerLevelFlowModel, sample_source
from midi_rae.utils import load_checkpoint

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'device: {device}')

## Checkpoint paths — adjust as needed

In [ ]:
HOME = Path(os.path.expanduser('~'))
RUNS  = HOME / 'runs/midi-rae'

PCA_DIR       = HOME / 'datasets/POP909_pca_exp26'
FLOW_CKPT     = RUNS / 'flow15_cross_LkAu9t/checkpoints/CrossLevelFlowModel_flow_CrossLevelFlowModel_best.pt'
HMEP_CKPT     = RUNS / 'lecun_hmep4_m9BX54/checkpoints/SwinMaskedEmbeddingPredictor_lecun_hmep4_m9BX54_best.pt'
DECODER_CKPT  = RUNS / 'dec26_2JIbRt/checkpoints/SwinDecoder_dec26_2JIbRt_best.pt'
ENCODER_CKPT  = HOME / 'runs/midi-rae/exp26_z1olvN/checkpoints/SwinEncoder_exp26_z1olvN_best.pt'

# Adjust FLOW_CKPT to whatever the latest best checkpoint is:
import glob
flow_ckpts = sorted(glob.glob(str(RUNS / 'flow15_cross_*/checkpoints/*_best.pt')))
if flow_ckpts: FLOW_CKPT = Path(flow_ckpts[-1])
print(f'Flow ckpt:    {FLOW_CKPT}')
print(f'HMEP ckpt:    {HMEP_CKPT}')
print(f'Decoder ckpt: {DECODER_CKPT}')

## Step 1 — Load PCA models and define inverse transform

In [ ]:
# Load sklearn PCA objects saved by fit_pca
pca_models = {}
for i in range(4):  # L0-L3
    path = PCA_DIR / f'pca_L{i}_n20.pkl'
    with open(path, 'rb') as f:
        pca_models[i] = pickle.load(f)
    print(f'L{i}: {pca_models[i].n_components_} components, '
          f'original dim={pca_models[i].n_features_in_}')

# Level dims in the flat flow model vector: [20, 80, 320, 1280]
# = n_patches_per_level * 20 PCA components
LEVEL_DIMS  = [20, 80, 320, 1280]   # L0-L3 PCA-projected dims
N_PATCHES   = [d // 20 for d in LEVEL_DIMS]  # [1, 4, 16, 64]
GRID_SIZES  = [int(n**0.5) for n in N_PATCHES]  # [1, 2, 4, 8]
ORIG_DIMS   = [pca_models[i].n_features_in_ for i in range(4)]  # per-patch orig dims
print(f'n_patches per level (L0-L3): {N_PATCHES}')
print(f'grid sizes:                   {GRID_SIZES}')
print(f'original embedding dims:      {ORIG_DIMS}')

# L4/L5 info (no PCA — these levels were excluded from flow model)
# embed_dim=8 doubles each stage: stage0=8,1=16,2=32,3=64,4=128,5=256
# Coarsest-first: L0=256-dim(1 patch), L1=128(4), L2=64(16), L3=32(64), L4=16(256), L5=8(1024)
L4_N_PATCHES, L4_DIM = 256, 16
L5_N_PATCHES, L5_DIM = 1024, 8

def make_grid_pos(n_patches, device='cpu'):
    """Construct (N, 2) grid position tensor for a square patch grid."""
    g = int(n_patches ** 0.5)
    rows, cols = torch.meshgrid(torch.arange(g), torch.arange(g), indexing='ij')
    return torch.stack([rows.flatten(), cols.flatten()], dim=1).float().to(device)

def inverse_pca_level(pca, flat_vec, n_patches, device):
    """Inverse PCA: flat (n_patches*20,) → (1, n_patches, orig_dim) PatchState emb."""
    pca_codes = flat_vec.reshape(n_patches, 20).cpu().numpy()  # (n_patches, 20)
    emb_np = pca.inverse_transform(pca_codes)                  # (n_patches, orig_dim)
    return torch.tensor(emb_np, dtype=torch.float32, device=device).unsqueeze(0)  # (1, N, D)

## Step 2 — Load flow model and sample L0-L3

In [ ]:
# Load flow model
flow_model = CrossLevelFlowModel(level_dims=LEVEL_DIMS, h_dim=512, n_layers=4,
                                  n_attn_layers=2, n_heads=8)
flow_model = load_checkpoint(flow_model, str(FLOW_CKPT))
flow_model = flow_model.to(device).eval()

source_scales = [8.0, 4.0, 2.0, 0.5]   # from config
source_df     = [None, None, None, 3.0] # from config (null=Gaussian, 3.0=Student-t)

def generate_l0_l3(n_samples=4, n_steps=100):
    """Sample L0-L3 embeddings via Euler integration of the flow ODE."""
    total_dim = sum(LEVEL_DIMS)  # 1700
    x = sample_source((n_samples, total_dim), device=device,
                      source_df=source_df, source_scales=source_scales,
                      level_dims=LEVEL_DIMS)
    dt = 1.0 / n_steps
    with torch.no_grad():
        for step in range(n_steps):
            t = torch.full((n_samples,), step * dt, device=device)
            v = flow_model(x, t)
            x = x + v * dt
    return x   # (n_samples, 1700)

N_SAMPLES = 4
flow_samples = generate_l0_l3(N_SAMPLES)
print(f'Flow output: {flow_samples.shape}  min={flow_samples.min():.2f}  max={flow_samples.max():.2f}')

## Step 3 — Inverse PCA: flow vectors → patch embeddings

In [ ]:
def build_patch_states_l0_l3(flow_vec, device):
    """Convert flat flow output (1700,) into list of PatchState for L0-L3."""
    states = []
    offset = 0
    for i, (n_patches, level_dim) in enumerate(zip(N_PATCHES, LEVEL_DIMS)):
        chunk = flow_vec[offset:offset + level_dim]   # (level_dim,)
        emb   = inverse_pca_level(pca_models[i], chunk, n_patches, device)  # (1, N, D)
        pos   = make_grid_pos(n_patches, device)                              # (N, 2)
        non_empty = torch.ones(1, n_patches, device=device)                  # (1, N)
        mae_mask  = torch.ones(n_patches, device=device)                     # (N,)
        states.append(PatchState(emb=emb, pos=pos, non_empty=non_empty, mae_mask=mae_mask))
        offset += level_dim
    return states   # [L0, L1, L2, L3]

# Build per-sample patch states for L0-L3
sample_patch_states = [build_patch_states_l0_l3(flow_samples[b], device)
                       for b in range(N_SAMPLES)]
print(f'L0 emb shape: {sample_patch_states[0][0].emb.shape}')   # (1, 1, 256)
print(f'L3 emb shape: {sample_patch_states[0][3].emb.shape}')   # (1, 64, 32)

## Step 4 — Load HMEP and predict L4/L5

In [ ]:
# Load encoder config to match HMEP dims
encoder = SwinEncoder(image_size=128, patch_h=4, patch_w=4, embed_dim=8,
                      depths=[2,2,2,6,2,1], num_heads=[2,2,2,4,8,16],
                      window_size=4, mlp_ratio=4.0, drop_path_rate=0.1)
# Embedding dims per level (coarsest-first): 256, 128, 64, 32, 16, 8
enc_dims = [int(8 * 2**(5-i)) for i in range(6)]  # [256, 128, 64, 32, 16, 8]
print(f'Encoder dims (coarsest→finest): {enc_dims}')

hmep = SwinMaskedEmbeddingPredictor(dims=enc_dims)
hmep = load_checkpoint(hmep, str(HMEP_CKPT))
hmep = hmep.to(device).eval()

def build_l45_patch_states(n_samples, device):
    """L4/L5 placeholder PatchStates with zeros — HMEP will fill these."""
    states = []
    for n_patches, dim in [(L4_N_PATCHES, L4_DIM), (L5_N_PATCHES, L5_DIM)]:
        pos       = make_grid_pos(n_patches, device)
        emb       = torch.zeros(n_samples, n_patches, dim, device=device)
        non_empty = torch.ones(n_samples, n_patches, device=device)
        mae_mask  = torch.ones(n_patches, device=device)
        states.append(PatchState(emb=emb, pos=pos, non_empty=non_empty, mae_mask=mae_mask))
    return states  # [L4, L5]

def predict_l45_with_hmep(l0_l3_states_per_sample, device):
    """Build batched EncoderOutput with generated L0-L3 + zero L4/L5,
    run HMEP with mask_ratio=0, return predicted L4/L5 embs."""
    B = len(l0_l3_states_per_sample)
    
    # Concatenate per-sample patch states into a batch
    def batch_states(per_sample_list):
        return PatchState(
            emb=torch.cat([s.emb for s in per_sample_list], dim=0),
            pos=per_sample_list[0].pos,  # same grid for all samples
            non_empty=torch.cat([s.non_empty for s in per_sample_list], dim=0),
            mae_mask=per_sample_list[0].mae_mask)
    
    l45 = build_l45_patch_states(B, device)
    
    # Batch across samples for each level
    batched_levels = []
    for li in range(4):  # L0-L3
        batched_levels.append(batch_states([s[li] for s in l0_l3_states_per_sample]))
    batched_levels.append(l45[0])  # L4 (already B-batched)
    batched_levels.append(l45[1])  # L5
    
    hps = HierarchicalPatchState(levels=batched_levels)
    # full_pos = finest level positions; full_non_empty = finest level non_empty
    enc_out = EncoderOutput(
        patches=hps,
        full_pos=batched_levels[-1].pos,
        full_non_empty=batched_levels[-1].non_empty,
        mae_mask=batched_levels[-1].mae_mask)
    
    with torch.no_grad():
        preds, _ = hmep(enc_out, mask_ratio=0)  # predict at all positions
    
    # preds[4] and preds[5] are the L4/L5 predictions
    return preds  # list of (B, N_i, D_i) per level

hmep_preds = predict_l45_with_hmep(sample_patch_states, device)
print(f'HMEP L4 pred shape: {hmep_preds[4].shape}')  # (B, 256, 16)
print(f'HMEP L5 pred shape: {hmep_preds[5].shape}')  # (B, 1024, 8)

## Step 5 — Assemble full EncoderOutput and decode

In [ ]:
# Load decoder
decoder = SwinDecoder(img_height=128, img_width=128, patch_h=4, patch_w=4,
                      out_channels=1, embed_dim=8,
                      depths=[2,2,2,6,2,1], num_heads=[2,2,2,4,8,16],
                      window_size=4, mlp_ratio=4.0, drop_path_rate=0.1)
decoder = load_checkpoint(decoder, str(DECODER_CKPT))
decoder = decoder.to(device).eval()

def build_full_enc_out(l0_l3_states_per_sample, hmep_preds, device):
    """Assemble final EncoderOutput with generated L0-L3 + HMEP L4/L5."""
    B = len(l0_l3_states_per_sample)
    levels = []
    
    # L0-L3: from flow model + inverse PCA (batch them)
    for li in range(4):
        embs = torch.cat([s[li].emb for s in l0_l3_states_per_sample], dim=0)
        pos  = l0_l3_states_per_sample[0][li].pos
        ne   = torch.cat([s[li].non_empty for s in l0_l3_states_per_sample], dim=0)
        mm   = l0_l3_states_per_sample[0][li].mae_mask
        levels.append(PatchState(emb=embs, pos=pos, non_empty=ne, mae_mask=mm))
    
    # L4/L5: from HMEP predictions
    for li, (n_patches, dim) in enumerate([(L4_N_PATCHES, L4_DIM), (L5_N_PATCHES, L5_DIM)]):
        pos = make_grid_pos(n_patches, device)
        ne  = torch.ones(B, n_patches, device=device)
        mm  = torch.ones(n_patches, device=device)
        levels.append(PatchState(emb=hmep_preds[4+li], pos=pos, non_empty=ne, mae_mask=mm))
    
    hps = HierarchicalPatchState(levels=levels)
    return EncoderOutput(
        patches=hps,
        full_pos=levels[-1].pos,
        full_non_empty=levels[-1].non_empty,
        mae_mask=levels[-1].mae_mask)

enc_out_full = build_full_enc_out(sample_patch_states, hmep_preds, device)

with torch.no_grad():
    recons = decoder(enc_out_full)  # (B, 1, 128, 128)
print(f'Reconstruction shape: {recons.shape}')

## Step 6 — Visualize generated piano rolls

In [ ]:
from midi_rae.utils import binarize

fig, axes = plt.subplots(1, N_SAMPLES, figsize=(4 * N_SAMPLES, 4))
for i, ax in enumerate(axes):
    img = binarize(recons[i, 0]).cpu().numpy()
    ax.imshow(img, aspect='auto', origin='lower', cmap='gray_r', vmin=0, vmax=1)
    ax.set_title(f'Sample {i+1}')
    ax.axis('off')
plt.suptitle('Generated Piano Rolls: Flow(L0-L3) → HMEP(L4-L5) → Decoder')
plt.tight_layout()
plt.savefig('generated_piano_rolls.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved generated_piano_rolls.png')

## Ablation: L4/L5 = zeros (no HMEP) vs HMEP prediction
Run this to compare what information L4/L5 actually contributes.

In [ ]:
def build_full_enc_out_zero_l45(l0_l3_states_per_sample, device):
    """Same as above but with all-zero L4/L5 (no HMEP)."""
    B = len(l0_l3_states_per_sample)
    levels = []
    for li in range(4):
        embs = torch.cat([s[li].emb for s in l0_l3_states_per_sample], dim=0)
        pos  = l0_l3_states_per_sample[0][li].pos
        ne   = torch.cat([s[li].non_empty for s in l0_l3_states_per_sample], dim=0)
        mm   = l0_l3_states_per_sample[0][li].mae_mask
        levels.append(PatchState(emb=embs, pos=pos, non_empty=ne, mae_mask=mm))
    for n_patches, dim in [(L4_N_PATCHES, L4_DIM), (L5_N_PATCHES, L5_DIM)]:
        pos = make_grid_pos(n_patches, device)
        levels.append(PatchState(
            emb=torch.zeros(B, n_patches, dim, device=device),
            pos=pos, non_empty=torch.ones(B, n_patches, device=device),
            mae_mask=torch.ones(n_patches, device=device)))
    hps = HierarchicalPatchState(levels=levels)
    return EncoderOutput(patches=hps, full_pos=levels[-1].pos,
                         full_non_empty=levels[-1].non_empty, mae_mask=levels[-1].mae_mask)

enc_out_zero = build_full_enc_out_zero_l45(sample_patch_states, device)
with torch.no_grad():
    recons_zero = decoder(enc_out_zero)

fig, axes = plt.subplots(2, N_SAMPLES, figsize=(4 * N_SAMPLES, 8))
for i in range(N_SAMPLES):
    axes[0, i].imshow(binarize(recons[i, 0]).cpu().numpy(),
                       aspect='auto', origin='lower', cmap='gray_r')
    axes[0, i].set_title(f'HMEP L4/L5 — {i+1}')
    axes[0, i].axis('off')
    axes[1, i].imshow(binarize(recons_zero[i, 0]).cpu().numpy(),
                       aspect='auto', origin='lower', cmap='gray_r')
    axes[1, i].set_title(f'Zero L4/L5 — {i+1}')
    axes[1, i].axis('off')
plt.suptitle('HMEP L4/L5 (top) vs Zero L4/L5 (bottom)')
plt.tight_layout()
plt.savefig('generated_ablation.png', dpi=150, bbox_inches='tight')
plt.show()